# Qwen Gradient Ascent

This notebook implements **standard Gradient Ascent (GA) unlearning only** for the trained Qwen3.5-2B classifier. It begins every deletion scenario from the same verified Original model, optimises only on authoritative training-forget rows, and uses the predefined retained-validation safeguard to select a model safely.

Completed Full Retraining outputs are read only during final behavioural evaluation. They never enter optimisation or checkpoint selection.


## 1. Gradient Ascent Method

### 1.1 Purpose

The experiment tests whether reversing the task loss on a deletion request can reduce learned behaviour associated with those rows while limiting damage to retained utility. The pretrained Qwen base remains frozen; only the already-trained LoRA adapter and binary classification head may change.


### 1.2 Gradient Ascent Objective

For trainable parameters $\theta$, Gradient Ascent minimises

$$
L_{GA}(\theta)=-L_{forget}(\theta).
$$

- $\theta$: the trainable LoRA and binary classification-head parameters;
- $L_{forget}$: ordinary, unweighted classification cross-entropy on authoritative training-forget examples.

Ordinary training minimises classification loss. Minimising its negative moves the optimiser in the opposite direction and therefore increases loss on examples that should be forgotten.


### 1.3 Why Retained-Utility Protection Is Needed

GA has no retained-data term, so unconstrained ascent can damage predictions outside the deletion request. Ordinary fitting uses $L=L_{data}$, while Gradient Difference uses $L_{GD}=L_{retain}-L_{forget}$. Standard GA instead uses forget data alone and relies on a separate retained-validation safeguard.

Retained validation is **evaluation-only**: it never contributes gradients. Its weighted CE may be at most 5% above the scenario-specific Original Qwen value.


### 1.4 Checkpoint Selection Rule

After each epoch, the complete forget-set CE and retained-validation weighted CE are evaluated. An updated checkpoint is eligible only when its parameters and evaluation values are finite and its validation CE is within the 5% boundary. Among eligible updates, the epoch with the greatest complete forget-set CE is selected.

Three consecutive safety breaches stop the trajectory. If no updated epoch is eligible, the Original epoch-0 model is restored and reported honestly. Retained-test metrics, Truth Ratio, KS, and Full Retraining similarity never select the checkpoint.


## 2. Environment Verification

The cells below verify the exact successful A100 environment. They deliberately do not install or upgrade packages. A mismatch stops execution before model loading or training.


### 2.1 GPU and CUDA


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version as package_version

import gc
import hashlib
import json
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch

from IPython.display import Markdown, display

# Training is intentionally restricted to the hardware used by the
# completed reference experiment, making runtime comparisons interpretable.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Use the final A100-SXM4-80GB RunPod; "
        "this notebook will not repair or replace the environment."
    )

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

gpu_checks = {
    "GPU is NVIDIA A100-SXM4-80GB": "A100-SXM4-80GB" in GPU_NAME,
    "PyTorch is 2.8.0+cu128": torch.__version__.startswith("2.8.0+cu128"),
    "Torch CUDA runtime is 12.8": torch.version.cuda == "12.8",
}

display(pd.DataFrame(
    {"Check": gpu_checks.keys(), "Passed": gpu_checks.values()}
).assign(Status=lambda frame: frame["Passed"].map({True: "PASS", False: "FAIL"})))

if not all(gpu_checks.values()):
    raise RuntimeError(
        f"Wrong GPU/CUDA environment: GPU={GPU_NAME!r}, "
        f"torch={torch.__version__!r}, CUDA={torch.version.cuda!r}. "
        "Stop here and use the successful final RunPod image."
    )


### 2.2 Package Versions


In [ ]:
# Unsloth must be imported before Transformers so its Qwen patches are active.
import unsloth
from unsloth import FastVisionModel

from transformers import AutoTokenizer
from peft import PeftModel
from scipy.stats import ks_2samp
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

EXPECTED_PACKAGES = {
    "unsloth": "2026.8.22",
    "transformers": "5.2.0",
    "triton": "3.4.0",
}

package_checks = {
    package: package_version(package) == expected
    for package, expected in EXPECTED_PACKAGES.items()
}

display(pd.DataFrame([
    {
        "Package": package,
        "Expected": expected,
        "Observed": package_version(package),
        "Status": "PASS" if package_checks[package] else "FAIL",
    }
    for package, expected in EXPECTED_PACKAGES.items()
]))

if not all(package_checks.values()):
    raise RuntimeError(
        "Package versions do not match the completed Full Retraining stack. "
        "Stop here; do not upgrade packages inside this notebook."
    )


### 2.3 Reproducibility


In [ ]:
SEED = 42

def reset_seed(seed=SEED):
    # Resetting all generators makes independently restored scenarios repeatable.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

reset_seed()
print("Global seed:", SEED)


## 3. Frozen Original Qwen

### 3.1 Locate Frozen Artefacts

The canonical final-submission bundle is authoritative. The search supports both the repository checkout and its RunPod mount without changing filenames or substituting an older run.


In [ ]:
def locate_final_submission():
    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
        Path("/workspace/qub-machine-unlearning"),
    ]

    for candidate in candidates:
        possible_roots = [
            candidate if candidate.name == "final_submission" else None,
            candidate / "final_submission",
            candidate / "code" / "final_submission",
        ]
        for root in possible_roots:
            if root is None:
                continue
            marker = root / "data" / "final" / "kidney_transplant_assessments.csv"
            if marker.is_file():
                return root.resolve()

    raise FileNotFoundError("Could not locate code/final_submission.")

FINAL = locate_final_submission()
MODEL_ROOT = FINAL / "models" / "qwen"
RESULT_ROOT = FINAL / "results" / "qwen"

BASELINE_ADAPTER = MODEL_ROOT / "baseline" / "adapter"
BASELINE_HEAD = MODEL_ROOT / "baseline" / "binary_classification_head.pt"
BASELINE_RESULTS = RESULT_ROOT / "original"
FULL_RESULTS = RESULT_ROOT / "full_retraining"

DATA_PATH = FINAL / "data" / "final" / "kidney_transplant_assessments.csv"
FEATURE_PATH = FINAL / "data" / "final" / "classifier_feature_list.json"
SPLIT_PATH = FINAL / "processed_data" / "split_assignments.csv"
MEMBERSHIP_PATH = FINAL / "processed_data" / "deletion_scenario_membership.csv"

METHOD_MODEL_ROOT = MODEL_ROOT / "gradient_ascent"
METHOD_RESULT_ROOT = RESULT_ROOT / "unlearning" / "gradient_ascent"


In [ ]:
required_inputs = {
    "assessment data": DATA_PATH,
    "feature contract": FEATURE_PATH,
    "permanent split": SPLIT_PATH,
    "deletion memberships": MEMBERSHIP_PATH,
    "adapter configuration": BASELINE_ADAPTER / "adapter_config.json",
    "classification head": BASELINE_HEAD,
    "baseline configuration": BASELINE_RESULTS / "experiment_configuration.json",
    "frozen threshold": BASELINE_RESULTS / "selected_threshold.json",
    "serialisation specification": BASELINE_RESULTS / "serialisation_specification.json",
    "saved baseline probabilities": BASELINE_RESULTS / "retained_test_probabilities.csv",
}

adapter_weights = [
    path for path in [
        BASELINE_ADAPTER / "adapter_model.safetensors",
        BASELINE_ADAPTER / "adapter_model.bin",
    ]
    if path.is_file()
]

missing_inputs = [path for path in required_inputs.values() if not path.is_file()]
if missing_inputs or len(adapter_weights) != 1:
    raise FileNotFoundError(
        "Frozen Original Qwen bundle is incomplete. Missing files:\n  "
        + "\n  ".join(map(str, missing_inputs))
        + f"\nAdapter weight candidates found: {adapter_weights}"
    )

BASELINE_ADAPTER_WEIGHTS = adapter_weights[0]
display(pd.DataFrame([
    {"Artefact": name, "Path": str(path), "Exists": path.is_file()}
    for name, path in required_inputs.items()
]))


### 3.2 Baseline Configuration


In [ ]:
baseline_config = json.loads(
    (BASELINE_RESULTS / "experiment_configuration.json").read_text(encoding="utf-8")
)
baseline_threshold = json.loads(
    (BASELINE_RESULTS / "selected_threshold.json").read_text(encoding="utf-8")
)
baseline_serialisation = json.loads(
    (BASELINE_RESULTS / "serialisation_specification.json").read_text(encoding="utf-8")
)
adapter_config = json.loads(
    (BASELINE_ADAPTER / "adapter_config.json").read_text(encoding="utf-8")
)

MODEL_ID = baseline_config["model_id"]
MAX_SEQ_LENGTH = int(baseline_config["max_seq_length"])
FROZEN_THRESHOLD = float(baseline_threshold["threshold"])

assert baseline_config["run_id"] == "20260829T151430Z"
assert MODEL_ID == "unsloth/Qwen3.5-2B-Base"
assert MAX_SEQ_LENGTH == 216
assert FROZEN_THRESHOLD == 0.55
assert int(adapter_config["r"]) == 16
assert int(adapter_config["lora_alpha"]) == 16
assert float(adapter_config["lora_dropout"]) == 0.0
assert adapter_config["base_model_name_or_path"] == MODEL_ID

display(pd.Series({
    "Baseline run": baseline_config["run_id"],
    "Model": MODEL_ID,
    "Maximum sequence length": MAX_SEQ_LENGTH,
    "Frozen threshold": FROZEN_THRESHOLD,
    "LoRA rank": adapter_config["r"],
    "LoRA alpha": adapter_config["lora_alpha"],
    "LoRA dropout": adapter_config["lora_dropout"],
}, name="Frozen value").to_frame())


### 3.3 Load Original Qwen

The function below follows the successful Full Retraining environment: 16-bit `FastVisionModel`, no 4-bit quantisation, and no gradient checkpointing. The two-class output layer is installed **before** loading the trained PEFT adapter, then the saved head is restored. This cell defines reconstruction; the model is instantiated once after the cached inputs are ready.


In [ ]:
class FP32ClassificationHead(nn.Linear):
    # Keeping the small classifier in FP32 avoids avoidable loss instability.
    def forward(self, hidden_states):
        return F.linear(hidden_states.float(), self.weight, self.bias)

def clear_device_cache():
    gc.collect()
    torch.cuda.empty_cache()

def load_original_qwen(tokenizer):
    reset_seed()
    clear_device_cache()

    base_model, _ = FastVisionModel.from_pretrained(
        MODEL_ID,
        load_in_4bit=False,
        load_in_16bit=True,
        max_seq_length=MAX_SEQ_LENGTH,
        use_gradient_checkpointing=False,
    )

    # The base represents pretrained knowledge and is outside this
    # adaptation-level unlearning experiment.
    for parameter in base_model.parameters():
        parameter.requires_grad = False

    old_head = base_model.get_output_embeddings()
    base_model.set_output_embeddings(FP32ClassificationHead(
        old_head.in_features,
        2,
        bias=False,
        device=old_head.weight.device,
        dtype=torch.float32,
    ))
    base_model.config.num_labels = 2
    base_model.config.pad_token_id = tokenizer.pad_token_id

    # is_trainable=True preserves the already trained adapter as the
    # starting point; it does not create a fresh random LoRA adapter.
    model = PeftModel.from_pretrained(
        base_model,
        BASELINE_ADAPTER,
        is_trainable=True,
    )

    head_state = torch.load(BASELINE_HEAD, map_location="cpu", weights_only=True)
    current_state = model.state_dict()
    bad_head_keys = [
        name for name, value in head_state.items()
        if name not in current_state or current_state[name].shape != value.shape
    ]
    if not head_state or bad_head_keys:
        raise RuntimeError(f"Binary-head restore failed: {bad_head_keys[:5]}")
    model.load_state_dict(head_state, strict=False)

    # Explicit permissions make the scientific scope auditable even if a
    # future PEFT release changes its default requires_grad behaviour.
    for name, parameter in model.named_parameters():
        parameter.requires_grad = ("lora_" in name or "lm_head" in name)
        if parameter.requires_grad:
            parameter.data = parameter.data.float()

    model.config.use_cache = False
    return model.to(DEVICE)


### 3.4 Verify Reconstructed Baseline


In [ ]:
def capture_trainable(model):
    # Only this compact state is copied between scenarios; the 2.2B base is not.
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }

def restore_trainable(model, state):
    named_parameters = dict(model.named_parameters())
    with torch.no_grad():
        for name, value in state.items():
            if name not in named_parameters:
                raise KeyError(f"Missing trainable parameter: {name}")
            named_parameters[name].copy_(value.to(named_parameters[name].device))

def state_fingerprint(state):
    digest = hashlib.sha256()
    for name, value in sorted(state.items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(str(tuple(tensor.shape)).encode("ascii"))
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


In [ ]:
def assert_trainable_scope(model):
    trainable_names = [
        name for name, parameter in model.named_parameters()
        if parameter.requires_grad
    ]
    forbidden = [
        name for name in trainable_names
        if "lora_" not in name and "lm_head" not in name
    ]

    assert trainable_names, "No trainable parameters were found."
    assert any("lora_" in name for name in trainable_names)
    assert any("lm_head" in name for name in trainable_names)
    assert not forbidden, f"Frozen Qwen base parameters became trainable: {forbidden[:5]}"
    return trainable_names


The actual 64-row reproduction check is run in Section 5.5, after the permanent test rows have been serialised and cached. It compares IDs, labels, and probabilities and performs no optimisation.


## 4. Dataset and Deletion Scenarios

### 4.1 Load the Frozen Dataset


In [ ]:
assessments = pd.read_csv(DATA_PATH)
feature_contract = json.loads(FEATURE_PATH.read_text(encoding="utf-8"))

TARGET = feature_contract["target"]
FEATURES = feature_contract["classifier_features"]

EXPECTED_FEATURES = [
    "recipient_age", "donor_age", "donor_type", "kidney_failure_cause",
    "previous_transplant", "dialysis_months", "abo_compatibility_category",
    "hla_mismatch_count", "antibody_risk_score", "cold_ischaemia_hours",
    "days_since_transplant", "creatinine_mg_dl", "creatinine_change_pct",
    "urine_output_ml_24h", "tacrolimus_level_ng_ml",
    "medication_adherence_pct", "infection_indicator", "previous_rejection",
]

assert len(assessments) == 60_000
assert assessments["assessment_id"].is_unique
assert TARGET == "acute_rejection_within_30_days"
assert FEATURES == EXPECTED_FEATURES
print("Frozen assessment rows:", len(assessments))


### 4.2 Recreate the Permanent Split


In [ ]:
split_assignments = pd.read_csv(SPLIT_PATH)

data = assessments.merge(
    split_assignments[["recipient_id", "donor_id", "split"]],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)
assert data["split"].notna().all()

split_frames = {
    split_name: data.loc[data["split"].eq(split_name)].copy().reset_index(drop=True)
    for split_name in ["train", "validation", "test"]
}
assert {name: len(frame) for name, frame in split_frames.items()} == {
    "train": 42_024,
    "validation": 8_988,
    "test": 8_988,
}

for frame in split_frames.values():
    frame["assessment_id"] = frame["assessment_id"].astype(str)
    frame["label"] = frame[TARGET].astype("int64")

display(pd.DataFrame([
    {"Split": name, "Rows": len(frame), "Positive prevalence": frame["label"].mean()}
    for name, frame in split_frames.items()
]))


### 4.3 Load Exact Deletion Membership


In [ ]:
# This file is frozen experimental input. No deletion rule is re-evaluated here.
membership = pd.read_csv(MEMBERSHIP_PATH)
membership["assessment_id"] = membership["assessment_id"].astype(str)

assert not membership.duplicated(["scenario", "assessment_id"]).any()

SCENARIO_LABELS = {
    "recipient_withdrawal": "Recipient Withdrawal",
    "donor_withdrawal": "Donor Withdrawal",
    "invalid_consent": "Invalid Consent",
    "hospital_removal": "Hospital Removal",
    "retention_expiry": "Retention Expiry",
}

# The original full order is retained solely for compatible sampling seeds.
ALL_SCENARIOS = [
    "recipient_withdrawal",
    "donor_withdrawal",
    "invalid_consent",
    "hospital_removal",
    "retention_expiry",
]
assert set(membership["scenario"]) == set(ALL_SCENARIOS)


### 4.4 Final Three Scenarios


In [ ]:
# These names are explicit so an accidental slice cannot change the experiment.
FINAL_SCENARIOS = [
    "recipient_withdrawal",
    "invalid_consent",
    "retention_expiry",
]

EXPECTED_FORGET = {
    "recipient_withdrawal": 426,
    "invalid_consent": 4_148,
    "retention_expiry": 6_262,
}

scenario_sets = {}
for scenario in FINAL_SCENARIOS:
    scenario_membership = membership.loc[membership["scenario"].eq(scenario)]
    membership_ids = {
        kind: set(scenario_membership.loc[
            scenario_membership["membership_type"].eq(kind), "assessment_id"
        ])
        for kind in ["training_forget", "deleted_validation", "deleted_test"]
    }

    training_forget = split_frames["train"].loc[
        split_frames["train"]["assessment_id"].isin(membership_ids["training_forget"])
    ].copy()
    retained_train = split_frames["train"].loc[
        ~split_frames["train"]["assessment_id"].isin(membership_ids["training_forget"])
    ].copy()
    retained_validation = split_frames["validation"].loc[
        ~split_frames["validation"]["assessment_id"].isin(membership_ids["deleted_validation"])
    ].copy()
    retained_test = split_frames["test"].loc[
        ~split_frames["test"]["assessment_id"].isin(membership_ids["deleted_test"])
    ].copy()

    scenario_sets[scenario] = {
        "training_forget": training_forget,
        "retained_train": retained_train,
        "retained_validation": retained_validation,
        "retained_test": retained_test,
        "deleted_validation_ids": membership_ids["deleted_validation"],
        "deleted_test_ids": membership_ids["deleted_test"],
    }


In [ ]:
scenario_audit = []
for scenario, parts in scenario_sets.items():
    forget_ids = set(parts["training_forget"]["assessment_id"])
    retain_ids = set(parts["retained_train"]["assessment_id"])

    assert len(forget_ids) == EXPECTED_FORGET[scenario]
    assert len(forget_ids) + len(retain_ids) == 42_024
    assert forget_ids.isdisjoint(retain_ids)

    scenario_audit.append({
        "Scenario": SCENARIO_LABELS[scenario],
        "Forget rows": len(forget_ids),
        "Retained training rows": len(retain_ids),
        "Total training rows": len(forget_ids) + len(retain_ids),
        "Disjoint": forget_ids.isdisjoint(retain_ids),
    })

display(pd.DataFrame(scenario_audit))


## 5. Qwen Input Preparation

### 5.1 Frozen Feature Order


In [ ]:
FEATURE_LABELS = baseline_serialisation["display_labels"]
BINARY_FEATURES = {
    "previous_transplant",
    "infection_indicator",
    "previous_rejection",
}

assert baseline_serialisation["feature_order"] == FEATURES
assert baseline_serialisation["target_included"] is False
assert baseline_serialisation["identifiers_included"] is False

display(pd.DataFrame({
    "Position": range(1, len(FEATURES) + 1),
    "Feature": FEATURES,
    "Display label": [FEATURE_LABELS[feature] for feature in FEATURES],
}))


### 5.2 Deterministic Text Serialisation


In [ ]:
def format_feature_value(feature, value):
    if pd.isna(value):
        return "missing"
    if feature in BINARY_FEATURES:
        return "yes" if int(value) == 1 else "no"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}".rstrip("0").rstrip(".")
    return str(value).strip()

def serialize_assessment(row):
    # Feature order is the saved model contract, not dataframe column order.
    return "\n".join(
        f"{FEATURE_LABELS[feature]}: {format_feature_value(feature, row[feature])}."
        for feature in FEATURES
    )


In [ ]:
# Serialise the complete dataset once so every scenario sees identical text.
serialised = data.copy()
serialised["assessment_id"] = serialised["assessment_id"].astype(str)
serialised["text"] = serialised.apply(serialize_assessment, axis=1)
text_by_id = serialised.set_index("assessment_id")["text"]

for parts in scenario_sets.values():
    for key in ["training_forget", "retained_train", "retained_validation", "retained_test"]:
        frame = parts[key]
        frame["text"] = frame["assessment_id"].map(text_by_id)
        assert frame["text"].notna().all()

print(serialised.loc[0, "text"])


### 5.3 Tokenise All Assessments Once


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_ADAPTER,
    local_files_only=True,
)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

TOKEN_CACHE = {}
TOKENISE_BATCH_SIZE = 2_048
all_ids = serialised["assessment_id"].tolist()
all_texts = serialised["text"].tolist()

for start in tqdm(range(0, len(all_texts), TOKENISE_BATCH_SIZE), desc="Tokenising once"):
    end = min(start + TOKENISE_BATCH_SIZE, len(all_texts))
    encoded = tokenizer(
        all_texts[start:end],
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    for assessment_id, token_ids in zip(all_ids[start:end], encoded["input_ids"]):
        # int32 halves cache storage relative to int64; collators cast only a batch.
        TOKEN_CACHE[assessment_id] = torch.tensor(token_ids, dtype=torch.int32)


In [ ]:
token_lengths = np.fromiter(
    (len(token_ids) for token_ids in TOKEN_CACHE.values()),
    dtype=np.int32,
    count=len(TOKEN_CACHE),
)

token_audit = pd.Series({
    "Cached rows": len(TOKEN_CACHE),
    "Maximum token length": int(token_lengths.max()),
    "Mean token length": float(token_lengths.mean()),
    "Frozen maximum sequence length": MAX_SEQ_LENGTH,
}, name="Value")
display(token_audit.to_frame())

assert len(TOKEN_CACHE) == 60_000
assert set(TOKEN_CACHE) == set(all_ids)
assert int(token_lengths.max()) <= MAX_SEQ_LENGTH


### 5.4 Cached DataLoader


In [ ]:
class CachedDataset(Dataset):
    def __init__(self, frame):
        frame = frame.reset_index(drop=True)
        self.ids = frame["assessment_id"].astype(str).tolist()
        self.labels = frame["label"].astype(int).tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        assessment_id = self.ids[index]
        return {
            "assessment_id": assessment_id,
            "input_ids": TOKEN_CACHE[assessment_id],
            "label": self.labels[index],
        }

def cached_collate(rows):
    # Dynamic padding avoids paying for 216 tokens when a batch is shorter.
    sequences = [row["input_ids"].long() for row in rows]
    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )
    return {
        "input_ids": input_ids,
        "attention_mask": (input_ids != tokenizer.pad_token_id).long(),
        "labels": torch.tensor([row["label"] for row in rows], dtype=torch.long),
        "assessment_id": [row["assessment_id"] for row in rows],
    }


In [ ]:
NUM_WORKERS = 2

def make_loader(frame, batch_size, shuffle=False, generator=None):
    return DataLoader(
        CachedDataset(frame),
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        collate_fn=cached_collate,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

def final_token_logits(model, batch):
    input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
    attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
    sequence_logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    ).logits
    final_indices = attention_mask.sum(dim=1) - 1
    logits = sequence_logits[
        torch.arange(input_ids.shape[0], device=DEVICE),
        final_indices,
    ]
    if logits.shape != (input_ids.shape[0], 2):
        raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")
    return logits


### 5.5 Load Once and Run the Baseline Diagnostic


In [ ]:
# This is the notebook's only 2.2B base-model load.
MODEL = load_original_qwen(tokenizer)
TRAINABLE_NAMES = assert_trainable_scope(MODEL)

FROZEN_BASELINE_STATE = capture_trainable(MODEL)
FROZEN_BASELINE_SHA256 = state_fingerprint(FROZEN_BASELINE_STATE)

display(pd.Series({
    "Total parameters": sum(p.numel() for p in MODEL.parameters()),
    "Trainable parameters": sum(p.numel() for p in MODEL.parameters() if p.requires_grad),
    "Trainable tensors": len(TRAINABLE_NAMES),
    "Baseline SHA-256": FROZEN_BASELINE_SHA256,
}, name="Value").to_frame())


In [ ]:
@torch.inference_mode()
def predict_fixed(model, frame, batch_size, description):
    model.eval()
    rows = []
    for batch in tqdm(make_loader(frame, batch_size), desc=description, leave=False):
        probabilities = torch.softmax(final_token_logits(model, batch).float(), dim=1)[:, 1]
        rows.extend({
            "assessment_id": assessment_id,
            "label": int(label),
            "probability_class_1": float(probability),
        } for assessment_id, label, probability in zip(
            batch["assessment_id"],
            batch["labels"].tolist(),
            probabilities.cpu().tolist(),
        ))
    return pd.DataFrame(rows)

def verify_reconstructed_baseline(sample_size=64):
    saved = pd.read_csv(BASELINE_RESULTS / "retained_test_probabilities.csv")
    saved["assessment_id"] = saved["assessment_id"].astype(str)
    expected = saved.head(sample_size).copy()
    expected_ids = expected["assessment_id"].tolist()

    indexed_test = split_frames["test"].set_index("assessment_id")
    if not set(expected_ids).issubset(indexed_test.index):
        raise RuntimeError("Saved baseline IDs are not in the permanent test split.")
    probe = indexed_test.loc[expected_ids].reset_index()
    observed = predict_fixed(MODEL, probe, 64, "Verify Original Qwen")

    differences = np.abs(
        observed["probability_class_1"].to_numpy()
        - expected["probability_class_1"].to_numpy()
    )
    report = {
        "Compared rows": len(observed),
        "IDs align": observed["assessment_id"].tolist() == expected_ids,
        "Labels align": np.array_equal(observed["label"], expected["label"]),
        "Maximum absolute probability difference": float(differences.max()),
        "Mean absolute probability difference": float(differences.mean()),
    }
    report["Passed"] = (
        report["IDs align"]
        and report["Labels align"]
        and np.allclose(
            observed["probability_class_1"],
            expected["probability_class_1"],
            atol=1e-5,
            rtol=1e-4,
        )
    )
    display(pd.Series(report, name="Baseline diagnostic").to_frame())
    if not report["Passed"]:
        raise RuntimeError(
            "Reconstructed Original Qwen materially disagrees with saved prediction evidence. "
            "Do not start unlearning."
        )
    return report

BASELINE_VERIFICATION = verify_reconstructed_baseline()


## 6. Gradient Ascent Setup

### 6.1 Prepare Forget Data

Only `training_forget` enters the optimisation loader. Its IDs must exactly equal the authoritative membership IDs for that scenario; no retained, validation, test, or Full Retraining row is eligible for an ascent update.


In [ ]:
def authoritative_forget_ids(scenario):
    scenario_membership = membership.loc[membership["scenario"].eq(scenario)]
    return set(scenario_membership.loc[
        scenario_membership["membership_type"].eq("training_forget"),
        "assessment_id",
    ])

for scenario in FINAL_SCENARIOS:
    optimiser_ids = set(scenario_sets[scenario]["training_forget"]["assessment_id"])
    assert optimiser_ids == authoritative_forget_ids(scenario)
    assert optimiser_ids.isdisjoint(
        set(scenario_sets[scenario]["retained_train"]["assessment_id"])
    )

print("Optimisation membership audit: PASS")


### 6.2 Prepare Retained Validation Data

Retained validation is used only after updates to enforce the safety boundary. Class weighting is derived from retained **training** labels, not from validation, test, forget, or Full Retraining data.


In [ ]:
def retained_training_positive_weight(frame):
    positives = int(frame["label"].sum())
    negatives = len(frame) - positives
    if positives == 0:
        raise RuntimeError("Retained training contains no positive examples.")
    return float(negatives / positives)

validation_weight_audit = pd.DataFrame([
    {
        "Scenario": SCENARIO_LABELS[scenario],
        "Weight source": "retained_train",
        "Retained training rows": len(scenario_sets[scenario]["retained_train"]),
        "Positive class weight": retained_training_positive_weight(
            scenario_sets[scenario]["retained_train"]
        ),
        "Retained validation rows": len(scenario_sets[scenario]["retained_validation"]),
    }
    for scenario in FINAL_SCENARIOS
])
display(validation_weight_audit)


### 6.3 Establish the Baseline Validation Loss


In [ ]:
def classification_cross_entropy(logits, labels, positive_weight=None):
    if positive_weight is None:
        return F.cross_entropy(logits.float(), labels)

    # The safety metric uses the established retained-training class ratio.
    weights = torch.tensor(
        [1.0, positive_weight], dtype=torch.float32, device=logits.device
    )
    return F.cross_entropy(
        logits.float(), labels, weight=weights, reduction="sum"
    ) / len(labels)


In [ ]:
EVALUATION_BATCHES = [256, 128, 64, 32]

@torch.inference_mode()
def evaluate_at_batch(model, frame, description, batch_size, positive_weight=None):
    model.eval()
    rows = []
    total_loss = 0.0
    total_rows = 0
    for batch in tqdm(make_loader(frame, batch_size), desc=description, leave=False):
        labels = batch["labels"].to(DEVICE, non_blocking=True)
        logits = final_token_logits(model, batch)
        loss = classification_cross_entropy(logits, labels, positive_weight)
        probabilities = torch.softmax(logits.float(), dim=1)[:, 1]
        batch_rows = len(labels)
        total_loss += float(loss.item()) * batch_rows
        total_rows += batch_rows
        rows.extend({
            "assessment_id": assessment_id,
            "label": int(label),
            "probability_class_1": float(probability),
        } for assessment_id, label, probability in zip(
            batch["assessment_id"],
            batch["labels"].tolist(),
            probabilities.cpu().tolist(),
        ))
    return pd.DataFrame(rows), total_loss / total_rows


In [ ]:
def evaluate_frame(model, frame, description, positive_weight=None):
    for batch_size in EVALUATION_BATCHES:
        try:
            predictions, mean_loss = evaluate_at_batch(
                model, frame, description, batch_size, positive_weight
            )
            return predictions, mean_loss, batch_size
        except torch.cuda.OutOfMemoryError:
            clear_device_cache()
            print(f"Evaluation OOM at {batch_size}; retrying with a smaller batch.")
    raise RuntimeError("Evaluation did not fit even at batch size 32.")


In [ ]:
def establish_safety_boundary(model, scenario):
    parts = scenario_sets[scenario]
    positive_weight = retained_training_positive_weight(parts["retained_train"])

    _, baseline_forget_ce, forget_batch = evaluate_frame(
        model, parts["training_forget"], f"Epoch 0 forget: {scenario}"
    )
    _, baseline_validation_ce, validation_batch = evaluate_frame(
        model,
        parts["retained_validation"],
        f"Epoch 0 retained validation: {scenario}",
        positive_weight,
    )
    validation_limit = baseline_validation_ce * 1.05
    display(pd.Series({
        "Baseline validation weighted CE": baseline_validation_ce,
        "Relative allowance": 0.05,
        "Absolute allowance": validation_limit - baseline_validation_ce,
        "Validation limit": validation_limit,
    }, name=SCENARIO_LABELS[scenario]).to_frame())
    return {
        "positive_weight": positive_weight,
        "baseline_forget_ce": baseline_forget_ce,
        "baseline_validation_weighted_ce": baseline_validation_ce,
        "validation_limit": validation_limit,
        "baseline_forget_evaluation_batch": forget_batch,
        "baseline_validation_evaluation_batch": validation_batch,
    }


### 6.4 Define the 5% Safety Boundary

For each scenario,

$$ValidationLimit=1.05	imes BaselineValidationLoss.$$

The baseline is re-evaluated after restoring the same Original state, because retained-validation membership differs by deletion request.


### 6.5 Gradient Ascent Configuration


In [ ]:
GA_CONFIG = {
    "method": "standard_gradient_ascent",
    "optimizer": "AdamW",
    "learning_rate": 1e-5,
    "weight_decay": 0.0,
    "physical_forget_batch": 32,
    "gradient_accumulation": 1,
    "effective_forget_batch": 32,
    "maximum_epochs": 20,
    "gradient_clip_norm": 1.0,
    "validation_relative_allowance": 0.05,
    "safety_patience": 3,
    "selection": "eligible epoch with greatest complete forget-set CE",
    "fallback": "epoch 0 when no safe updated checkpoint exists",
    "baseline_run_id": "20260829T151430Z",
}

assert GA_CONFIG["effective_forget_batch"] == 32
assert GA_CONFIG["maximum_epochs"] == 20
assert GA_CONFIG["safety_patience"] == 3
assert GA_CONFIG["validation_relative_allowance"] == 0.05

GA_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(GA_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()
display(pd.Series(GA_CONFIG, name="Frozen GA value").to_frame())


In [ ]:
training_scale = pd.DataFrame([
    {
        "Scenario": SCENARIO_LABELS[scenario],
        "Forget rows": EXPECTED_FORGET[scenario],
        "Approximate update batches per epoch": int(np.ceil(
            EXPECTED_FORGET[scenario] / GA_CONFIG["physical_forget_batch"]
        )),
    }
    for scenario in FINAL_SCENARIOS
])
display(training_scale)


GA update cost scales mainly with deletion-request size: approximately 14, 130, and 196 update batches per epoch. Required complete-forget and retained-validation inference adds method runtime, especially when the update itself is short. Batch 32 with no accumulation preserves the established effective batch while removing avoidable accumulation overhead.


## 7. Gradient Ascent Implementation

### 7.1 Forget-Set Training Loader


In [ ]:
def make_forget_loader(scenario, epoch):
    forget_frame = scenario_sets[scenario]["training_forget"]
    loader_ids = set(forget_frame["assessment_id"])

    # This assertion is next to loader creation so a future edit cannot
    # silently introduce retained or evaluation rows into optimisation.
    assert loader_ids == authoritative_forget_ids(scenario)
    return make_loader(
        forget_frame,
        batch_size=GA_CONFIG["physical_forget_batch"],
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED + epoch),
    )


### 7.2 Forget Loss

The update uses ordinary unweighted cross-entropy over training-forget examples only.


### 7.3 Gradient Ascent Objective

The central operation remains visible in both the smoke test and training loop:

```python
forget_loss = classification_cross_entropy(forget_logits, forget_labels)
objective = -forget_loss
```


In [ ]:
def require_finite(value, name, scenario, epoch, step):
    if not bool(torch.isfinite(value).all().item()):
        raise RuntimeError(
            f"Non-finite {name}: scenario={scenario}, epoch={epoch}, step={step}."
        )

def assert_finite_trainable_parameters(model, scenario, epoch):
    bad_names = [
        name for name, parameter in model.named_parameters()
        if parameter.requires_grad and not bool(torch.isfinite(parameter).all().item())
    ]
    if bad_names:
        raise RuntimeError(
            f"Non-finite trainable parameters after {scenario} epoch {epoch}: {bad_names[:5]}"
        )
    return True


### 7.4 Retained-Validation Safeguard

Retained validation receives no backward pass. It is evaluated after each epoch with weights derived solely from retained training. This monitors utility without turning GA into Gradient Difference or Retain-Set Fine-Tuning.


### 7.5 Eligible Checkpoint Rule

Eligibility requires finite parameters, finite complete forget CE, finite retained-validation CE, and validation CE no greater than the scenario limit. The selected updated checkpoint is the eligible epoch with greatest complete forget CE. Consecutive breaches reset to zero after any eligible epoch.


### 7.6 Verify the Objective


### 7.7 Smoke Test


In [ ]:
def run_ga_smoke_test():
    scenario = "recipient_withdrawal"
    restore_trainable(MODEL, FROZEN_BASELINE_STATE)
    assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256
    assert_trainable_scope(MODEL)

    loader = make_forget_loader(scenario, epoch=1)
    batch = next(iter(loader))
    assert set(batch["assessment_id"]).issubset(authoritative_forget_ids(scenario))
    parameters = [p for p in MODEL.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        parameters,
        lr=GA_CONFIG["learning_rate"],
        weight_decay=GA_CONFIG["weight_decay"],
    )

    MODEL.train()
    forget_labels = batch["labels"].to(DEVICE, non_blocking=True)
    forget_logits = final_token_logits(MODEL, batch)
    forget_loss = classification_cross_entropy(forget_logits, forget_labels)

    # Minimising this negative loss performs Gradient Ascent.
    objective = -forget_loss

    require_finite(forget_loss, "forget loss", scenario, 0, 1)
    require_finite(objective, "GA objective", scenario, 0, 1)
    objective.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        parameters, GA_CONFIG["gradient_clip_norm"]
    )
    require_finite(grad_norm, "gradient norm", scenario, 0, 1)
    optimizer.step()

    report = {
        "Ordinary forget CE": float(forget_loss.item()),
        "GA objective": float(objective.item()),
        "Gradient norm": float(grad_norm.item()),
    }
    assert np.isclose(
        report["GA objective"], -report["Ordinary forget CE"], atol=1e-7, rtol=0
    )

    # The disposable update must have zero effect on final scenario runs.
    restore_trainable(MODEL, FROZEN_BASELINE_STATE)
    MODEL.zero_grad(set_to_none=True)
    assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256
    del optimizer
    clear_device_cache()
    display(pd.Series(report, name="Disposable GA smoke test").to_frame())

run_ga_smoke_test()


### 7.8 Safe Output Paths


In [ ]:
def scenario_paths(scenario):
    model_dir = METHOD_MODEL_ROOT / scenario
    result_dir = METHOD_RESULT_ROOT / scenario
    return {
        "model_dir": model_dir,
        "adapter": model_dir / "adapter",
        "head": model_dir / "binary_classification_head.pt",
        "result_dir": result_dir,
        "checkpoint": result_dir / "_resume_checkpoint.pt",
        "complete": result_dir / "COMPLETE.json",
    }

def valid_complete_payload(payload, scenario):
    return (
        payload.get("status") == "complete"
        and payload.get("scenario") == scenario
        and payload.get("method") == GA_CONFIG["method"]
        and payload.get("baseline_sha256") == FROZEN_BASELINE_SHA256
        and payload.get("configuration_sha256") == GA_CONFIG_SHA256
    )


In [ ]:
def archive_partial(paths, scenario, reason):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive = METHOD_RESULT_ROOT / "_partial_archive" / stamp / scenario
    archive.mkdir(parents=True, exist_ok=False)
    for source in [paths["model_dir"], paths["result_dir"]]:
        if source.exists():
            shutil.move(str(source), str(archive / source.name))
    print(f"Archived incomplete/stale run ({reason}): {archive}")

def prepare_scenario(scenario):
    paths = scenario_paths(scenario)
    if paths["complete"].is_file():
        payload = json.loads(paths["complete"].read_text(encoding="utf-8"))
        if valid_complete_payload(payload, scenario):
            return "skip"
        archive_partial(paths, scenario, "invalid COMPLETE marker")
    elif paths["checkpoint"].is_file():
        checkpoint = torch.load(paths["checkpoint"], map_location="cpu", weights_only=False)
        valid_checkpoint = (
            checkpoint.get("scenario") == scenario
            and checkpoint.get("method") == GA_CONFIG["method"]
            and checkpoint.get("baseline_sha256") == FROZEN_BASELINE_SHA256
            and checkpoint.get("configuration_sha256") == GA_CONFIG_SHA256
            and 1 <= int(checkpoint.get("epoch", -1)) <= GA_CONFIG["maximum_epochs"]
        )
        if valid_checkpoint:
            return "resume"
        archive_partial(paths, scenario, "incompatible resume checkpoint")
    elif paths["model_dir"].exists() or paths["result_dir"].exists():
        archive_partial(paths, scenario, "no valid completion/checkpoint marker")

    paths["model_dir"].mkdir(parents=True, exist_ok=True)
    paths["result_dir"].mkdir(parents=True, exist_ok=True)
    return "run"


### 7.9 Lightweight Resume Checkpoints


In [ ]:
def save_resume_checkpoint(path, state):
    # The checkpoint contains only LoRA/head state, never the frozen Qwen base.
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, temporary)
    temporary.replace(path)

def optimizer_to_device(optimizer):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)


### 7.10 Initialise or Resume a Scenario


In [ ]:
def initialise_ga_training(model, scenario, resume):
    paths = scenario_paths(scenario)
    restore_trainable(model, FROZEN_BASELINE_STATE)
    assert state_fingerprint(capture_trainable(model)) == FROZEN_BASELINE_SHA256
    assert_trainable_scope(model)

    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        parameters,
        lr=GA_CONFIG["learning_rate"],
        weight_decay=GA_CONFIG["weight_decay"],
    )
    if not resume:
        return {
            "paths": paths,
            "parameters": parameters,
            "optimizer": optimizer,
            "start_epoch": 1,
            "best_state": None,
            "best_forget_ce": -np.inf,
            "selected_epoch": 0,
            "consecutive_breaches": 0,
            "history": [],
            "ga_update_seconds": 0.0,
            "ga_total_required_method_seconds": 0.0,
            "safety": None,
        }

    checkpoint = torch.load(paths["checkpoint"], map_location="cpu", weights_only=False)
    restore_trainable(model, checkpoint["current_trainable_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    optimizer_to_device(optimizer)
    display(pd.Series({
        "Baseline validation weighted CE": checkpoint["safety"]["baseline_validation_weighted_ce"],
        "Relative allowance": GA_CONFIG["validation_relative_allowance"],
        "Validation limit": checkpoint["safety"]["validation_limit"],
    }, name=f"Resumed {SCENARIO_LABELS[scenario]}").to_frame())
    return {
        "paths": paths,
        "parameters": parameters,
        "optimizer": optimizer,
        "start_epoch": int(checkpoint["epoch"]) + 1,
        "best_state": checkpoint["best_eligible_trainable_state"],
        "best_forget_ce": checkpoint["highest_eligible_forget_ce"],
        "selected_epoch": checkpoint["selected_epoch_so_far"],
        "consecutive_breaches": checkpoint["consecutive_safety_breaches"],
        "history": checkpoint["history"],
        "ga_update_seconds": checkpoint["ga_update_seconds"],
        "ga_total_required_method_seconds": checkpoint["ga_total_required_method_seconds"],
        "safety": checkpoint["safety"],
    }


### 7.11 One Forget-Only Ascent Epoch


In [ ]:
def run_ga_updates(model, scenario, epoch, optimizer, parameters):
    forget_loader = make_forget_loader(scenario, epoch)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_forget_ce = 0.0
    rows_seen = 0

    torch.cuda.synchronize()
    started = time.perf_counter()
    for step, forget_batch in enumerate(
        tqdm(forget_loader, desc=f"GA {scenario} epoch {epoch}"), 1
    ):
        assert set(forget_batch["assessment_id"]).issubset(
            authoritative_forget_ids(scenario)
        )
        forget_labels = forget_batch["labels"].to(DEVICE, non_blocking=True)
        forget_logits = final_token_logits(model, forget_batch)

        # Ordinary classification loss on data that should be forgotten.
        forget_loss = classification_cross_entropy(forget_logits, forget_labels)

        # Minimising the negative loss performs Gradient Ascent.
        objective = -forget_loss

        require_finite(forget_loss, "forget loss", scenario, epoch, step)
        require_finite(objective, "GA objective", scenario, epoch, step)
        objective.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            parameters, GA_CONFIG["gradient_clip_norm"]
        )
        require_finite(grad_norm, "gradient norm", scenario, epoch, step)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        total_forget_ce += float(forget_loss.detach().item()) * len(forget_labels)
        rows_seen += len(forget_labels)

    torch.cuda.synchronize()
    update_seconds = time.perf_counter() - started
    if rows_seen != len(scenario_sets[scenario]["training_forget"]):
        raise RuntimeError("GA epoch did not cover the complete forget request.")
    return total_forget_ce / rows_seen, update_seconds


### 7.12 Post-Epoch Safety Evaluation


In [ ]:
def evaluate_ga_epoch(model, scenario, epoch, safety):
    parts = scenario_sets[scenario]
    # Check parameters before paying for two full evaluation passes.
    model_finite = assert_finite_trainable_parameters(model, scenario, epoch)
    _, complete_forget_ce, forget_batch = evaluate_frame(
        model,
        parts["training_forget"],
        f"Complete forget: {scenario} epoch {epoch}",
    )
    _, validation_ce, validation_batch = evaluate_frame(
        model,
        parts["retained_validation"],
        f"Retained validation: {scenario} epoch {epoch}",
        safety["positive_weight"],
    )
    forget_finite = bool(np.isfinite(complete_forget_ce))
    validation_finite = bool(np.isfinite(validation_ce))
    if not forget_finite:
        raise RuntimeError(f"Non-finite complete forget CE: {scenario}, epoch {epoch}")
    if not validation_finite:
        raise RuntimeError(f"Non-finite retained-validation CE: {scenario}, epoch {epoch}")

    eligible = bool(
        model_finite
        and forget_finite
        and validation_finite
        and validation_ce <= safety["validation_limit"]
    )
    return {
        "complete_forget_ce": complete_forget_ce,
        "retained_validation_weighted_ce": validation_ce,
        "eligible": eligible,
        "forget_evaluation_batch": forget_batch,
        "validation_evaluation_batch": validation_batch,
    }


### 7.13 Select Eligible Checkpoints and Apply Safety Patience


In [ ]:
def update_ga_selection(model, state, scenario, epoch, mean_ce, evaluation,
                        epoch_update_seconds, epoch_runtime_seconds):
    if evaluation["eligible"] and evaluation["complete_forget_ce"] > state["best_forget_ce"]:
        state["best_state"] = capture_trainable(model)
        state["best_forget_ce"] = evaluation["complete_forget_ce"]
        state["selected_epoch"] = epoch

    state["consecutive_breaches"] = (
        0 if evaluation["eligible"] else state["consecutive_breaches"] + 1
    )
    safety = state["safety"]
    row = {
        "scenario": scenario,
        "epoch": epoch,
        "mean_batch_forget_ce": mean_ce,
        "complete_forget_ce": evaluation["complete_forget_ce"],
        "retained_validation_weighted_ce": evaluation["retained_validation_weighted_ce"],
        "baseline_retained_validation_weighted_ce": safety["baseline_validation_weighted_ce"],
        "validation_safety_limit": safety["validation_limit"],
        "validation_relative_change": (
            evaluation["retained_validation_weighted_ce"]
            / safety["baseline_validation_weighted_ce"] - 1
        ),
        "eligible": evaluation["eligible"],
        "highest_eligible_forget_ce_so_far": (
            state["best_forget_ce"] if state["best_state"] is not None else np.nan
        ),
        "selected_epoch_so_far": state["selected_epoch"],
        "consecutive_safety_breaches": state["consecutive_breaches"],
        "forget_evaluation_batch": evaluation["forget_evaluation_batch"],
        "validation_evaluation_batch": evaluation["validation_evaluation_batch"],
        "epoch_update_seconds": epoch_update_seconds,
        "epoch_runtime_seconds": epoch_runtime_seconds,
    }
    state["history"].append(row)
    display(pd.DataFrame([row]).round(6))


### 7.14 Save Each Completed Epoch


In [ ]:
def checkpoint_ga_epoch(model, scenario, epoch, state):
    save_resume_checkpoint(state["paths"]["checkpoint"], {
        "method": GA_CONFIG["method"],
        "scenario": scenario,
        "epoch": epoch,
        "current_trainable_state": capture_trainable(model),
        "best_eligible_trainable_state": state["best_state"],
        "selected_epoch_so_far": state["selected_epoch"],
        "highest_eligible_forget_ce": state["best_forget_ce"],
        "consecutive_safety_breaches": state["consecutive_breaches"],
        "baseline_validation_weighted_ce": state["safety"]["baseline_validation_weighted_ce"],
        "validation_limit": state["safety"]["validation_limit"],
        "optimizer_state": state["optimizer"].state_dict(),
        "history": state["history"],
        "ga_update_seconds": state["ga_update_seconds"],
        "ga_total_required_method_seconds": state["ga_total_required_method_seconds"],
        "baseline_sha256": FROZEN_BASELINE_SHA256,
        "configuration": GA_CONFIG,
        "configuration_sha256": GA_CONFIG_SHA256,
        "safety": state["safety"],
    })


### 7.15 Complete the GA Trajectory


In [ ]:
def train_ga(model, scenario, resume=False):
    state = initialise_ga_training(model, scenario, resume)

    if state["safety"] is None:
        # Baseline safety inference is required by GA selection and is included
        # in practical method time, but not in update-only time.
        torch.cuda.synchronize()
        safety_started = time.perf_counter()
        state["safety"] = establish_safety_boundary(model, scenario)
        torch.cuda.synchronize()
        state["ga_total_required_method_seconds"] += time.perf_counter() - safety_started

    torch.cuda.reset_peak_memory_stats()
    for epoch in range(state["start_epoch"], GA_CONFIG["maximum_epochs"] + 1):
        if state["consecutive_breaches"] >= GA_CONFIG["safety_patience"]:
            break

        torch.cuda.synchronize()
        epoch_started = time.perf_counter()
        mean_ce, update_seconds = run_ga_updates(
            model, scenario, epoch, state["optimizer"], state["parameters"]
        )
        evaluation = evaluate_ga_epoch(model, scenario, epoch, state["safety"])
        torch.cuda.synchronize()
        epoch_runtime = time.perf_counter() - epoch_started

        state["ga_update_seconds"] += update_seconds
        state["ga_total_required_method_seconds"] += epoch_runtime
        update_ga_selection(
            model, state, scenario, epoch, mean_ce, evaluation,
            update_seconds, epoch_runtime,
        )
        checkpoint_ga_epoch(model, scenario, epoch, state)

        if state["consecutive_breaches"] >= GA_CONFIG["safety_patience"]:
            print("Safety stop: three consecutive ineligible epochs.")
            break

    return finalise_ga_training(model, scenario, state)


In [ ]:
def finalise_ga_training(model, scenario, state):
    eligible_updated_model = state["best_state"] is not None
    selected_state = (
        state["best_state"] if eligible_updated_model else FROZEN_BASELINE_STATE
    )
    restore_trainable(model, selected_state)
    if not eligible_updated_model:
        print("No safe updated checkpoint: restored Original Qwen (epoch 0).")

    safety = state["safety"]
    summary = {
        **GA_CONFIG,
        "scenario": scenario,
        "selected_epoch": int(state["selected_epoch"]),
        "eligible_updated_model": bool(eligible_updated_model),
        "epoch_zero_fallback": not eligible_updated_model,
        "baseline_forget_ce": float(safety["baseline_forget_ce"]),
        "selected_forget_ce": float(
            state["best_forget_ce"] if eligible_updated_model
            else safety["baseline_forget_ce"]
        ),
        "baseline_validation_weighted_ce": float(safety["baseline_validation_weighted_ce"]),
        "validation_limit": float(safety["validation_limit"]),
        "baseline_forget_evaluation_batch": safety["baseline_forget_evaluation_batch"],
        "baseline_validation_evaluation_batch": safety["baseline_validation_evaluation_batch"],
        "epochs_executed": int(state["history"][-1]["epoch"]) if state["history"] else 0,
        "safety_stop_triggered": state["consecutive_breaches"] >= GA_CONFIG["safety_patience"],
        "ga_update_seconds": float(state["ga_update_seconds"]),
        "ga_total_required_method_seconds": float(state["ga_total_required_method_seconds"]),
        "peak_gpu_memory_gib": float(torch.cuda.max_memory_allocated() / 1024**3),
        "gpu": GPU_NAME,
        "baseline_sha256": FROZEN_BASELINE_SHA256,
        "configuration_sha256": GA_CONFIG_SHA256,
    }
    return state["history"], summary


## 8. Evaluation

The selected eligible checkpoint—or epoch 0 fallback—is restored before this section runs. Evaluation is post-selection evidence only.

### 8.1 Retained-Test Utility


In [ ]:
def utility_metrics(predictions):
    labels = predictions["label"].to_numpy()
    probabilities = predictions["probability_class_1"].to_numpy()
    predicted = (probabilities >= FROZEN_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "n": len(labels),
        "pr_auc": average_precision_score(labels, probabilities),
        "balanced_accuracy": balanced_accuracy_score(labels, predicted),
        "binary_cross_entropy": log_loss(labels, probabilities, labels=[0, 1]),
        "f1": f1_score(labels, predicted, zero_division=0),
        "auroc": roc_auc_score(labels, probabilities),
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "specificity": tn / (tn + fp),
        "threshold": FROZEN_THRESHOLD,
    }


### 8.2 Forget-Set Probabilities

The selected model is evaluated on the complete authoritative training-forget request.


### 8.3 Truth Ratio

For true-label probability $p_{true}$, Truth Ratio is

$$\frac{p_{incorrect}+10^{-12}}{p_{true}+10^{-12}}.$$


In [ ]:
def truth_components(labels, probabilities, epsilon=1e-12):
    p_true = np.where(labels == 1, probabilities, 1 - probabilities)
    p_incorrect = 1 - p_true
    truth_ratio = (p_incorrect + epsilon) / (p_true + epsilon)
    return p_true, p_incorrect, truth_ratio


### 8.4 KS Comparison with Full Retraining

Lower KS indicates closer observed Truth Ratio distributions. The p-value is supporting evidence; a high value alone does not prove forgetting. Behavioural similarity does not establish parameter-level erasure or legal compliance, and neither KS nor Full Retraining affects selection.


In [ ]:
def load_full_retraining_reference(scenario):
    result_dir = FULL_RESULTS / scenario
    complete_path = result_dir / "COMPLETE.json"
    forget_path = result_dir / "forget_set_probabilities.csv"
    if not complete_path.is_file() or not forget_path.is_file():
        raise FileNotFoundError(f"Missing Full Retraining reference: {result_dir}")
    complete = json.loads(complete_path.read_text(encoding="utf-8"))
    if complete.get("status") != "complete":
        raise RuntimeError(f"Full Retraining is incomplete: {scenario}")
    forget = pd.read_csv(forget_path)
    forget["assessment_id"] = forget["assessment_id"].astype(str)
    return complete, forget


In [ ]:
def forgetting_evidence(ga_predictions, full_predictions):
    ga_predictions = ga_predictions.copy()
    full_predictions = full_predictions.copy()
    ga_predictions["assessment_id"] = ga_predictions["assessment_id"].astype(str)
    full_predictions["assessment_id"] = full_predictions["assessment_id"].astype(str)
    assert set(ga_predictions["assessment_id"]) == set(full_predictions["assessment_id"])

    joined = ga_predictions.merge(
        full_predictions,
        on="assessment_id",
        suffixes=("_ga", "_full"),
        validate="one_to_one",
    )
    assert len(joined) == len(ga_predictions) == len(full_predictions)
    assert np.array_equal(joined["label_ga"], joined["label_full"])

    labels = joined["label_ga"].to_numpy()
    ga_probability = joined["probability_class_1_ga"].to_numpy()
    full_probability = joined["probability_class_1_full"].to_numpy()
    ga_true, ga_incorrect, ga_ratio = truth_components(labels, ga_probability)
    full_true, full_incorrect, full_ratio = truth_components(labels, full_probability)
    ks_result = ks_2samp(ga_ratio, full_ratio, alternative="two-sided", method="auto")
    values = pd.DataFrame({
        "assessment_id": joined["assessment_id"],
        "label": labels,
        "ga_probability": ga_probability,
        "full_retraining_probability": full_probability,
        "ga_p_true": ga_true,
        "ga_p_incorrect": ga_incorrect,
        "ga_truth_ratio": ga_ratio,
        "full_retraining_p_true": full_true,
        "full_retraining_p_incorrect": full_incorrect,
        "full_retraining_truth_ratio": full_ratio,
        "epsilon": 1e-12,
    })
    return values, {
        "forget_rows": len(values),
        "ks_statistic": float(ks_result.statistic),
        "ks_p_value": float(ks_result.pvalue),
    }


### 8.5 Computational Runtime

`ga_update_seconds` records ascent updates only. `ga_total_required_method_seconds` includes the scenario-specific baseline safeguard, ascent updates, complete-forget evaluations, and retained-validation safety evaluations, while excluding model loading, compilation, global tokenisation, final test/forget reporting, and file saving. Formal speed-up uses this practical required-method time; it is conservative relative to a training-only Full Retraining denominator, and the boundary is saved explicitly.


### 8.6 Save the Selected Model and Evidence


In [ ]:
def save_selected_model(model, scenario):
    paths = scenario_paths(scenario)
    paths["adapter"].mkdir(parents=True, exist_ok=True)
    model.save_pretrained(paths["adapter"], safe_serialization=True)
    tokenizer.save_pretrained(paths["adapter"])
    head_state = {
        name: value.detach().cpu()
        for name, value in model.state_dict().items()
        if "lm_head" in name
    }
    if not head_state:
        raise RuntimeError("Binary classification head was not found while saving.")
    torch.save(head_state, paths["head"])


In [ ]:
def save_evaluation_tables(paths, history, metrics, retained_predictions,
                           forget_predictions, truth_values, forgetting):
    pd.DataFrame(history).to_csv(paths["result_dir"] / "training_history.csv", index=False)
    pd.DataFrame([metrics]).to_csv(paths["result_dir"] / "retained_test_metrics.csv", index=False)
    retained_predictions.to_csv(
        paths["result_dir"] / "retained_test_probabilities.csv", index=False
    )
    forget_predictions.to_csv(
        paths["result_dir"] / "forget_set_probabilities.csv", index=False
    )
    truth_values.to_csv(paths["result_dir"] / "truth_ratio_values.csv", index=False)
    pd.DataFrame([forgetting]).to_csv(
        paths["result_dir"] / "forgetting_metrics.csv", index=False
    )


In [ ]:
def evaluate_selected_ga(model, scenario, summary):
    parts = scenario_sets[scenario]
    retained_predictions, _, retained_batch = evaluate_frame(
        model, parts["retained_test"], f"GA retained test: {scenario}"
    )
    forget_predictions, _, forget_batch = evaluate_frame(
        model, parts["training_forget"], f"GA forget set: {scenario}"
    )
    metrics = utility_metrics(retained_predictions)
    full_complete, full_forget = load_full_retraining_reference(scenario)
    truth_values, forgetting = forgetting_evidence(forget_predictions, full_forget)
    full_seconds = float(full_complete["training_seconds"])
    runtime = {
        "ga_update_seconds": summary["ga_update_seconds"],
        "ga_total_required_method_seconds": summary["ga_total_required_method_seconds"],
        "full_retraining_training_seconds": full_seconds,
        "formal_speed_up": full_seconds / summary["ga_total_required_method_seconds"],
        "formal_speed_up_ga_boundary": "GA required method time including safety selection",
        "full_retraining_boundary": "saved Full Retraining training time",
        "gpu": GPU_NAME,
        "retained_test_evaluation_batch": retained_batch,
        "forget_evaluation_batch": forget_batch,
    }
    return retained_predictions, forget_predictions, metrics, truth_values, forgetting, runtime


In [ ]:
def publish_ga_result(model, scenario, history, summary, evidence):
    paths = scenario_paths(scenario)
    retained, forget, metrics, truth_values, forgetting, runtime = evidence
    save_selected_model(model, scenario)
    save_evaluation_tables(
        paths, history, metrics, retained, forget, truth_values, forgetting
    )
    (paths["result_dir"] / "runtime.json").write_text(
        json.dumps(runtime, indent=2), encoding="utf-8"
    )
    (paths["result_dir"] / "configuration.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )

    complete = {
        "status": "complete",
        "method": GA_CONFIG["method"],
        "scenario": scenario,
        **summary,
        **metrics,
        **forgetting,
        **runtime,
        "training_forget_rows": len(forget),
        "retained_test_rows": len(retained),
    }
    required = [
        paths["adapter"] / "adapter_config.json",
        paths["head"],
        *[paths["result_dir"] / name for name in [
            "training_history.csv", "retained_test_metrics.csv",
            "retained_test_probabilities.csv", "forget_set_probabilities.csv",
            "truth_ratio_values.csv", "forgetting_metrics.csv",
            "runtime.json", "configuration.json",
        ]],
    ]
    missing = [path for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(f"Refusing COMPLETE marker; missing outputs: {missing}")

    # COMPLETE is the final atomic publication step.
    temporary = paths["complete"].with_suffix(".json.tmp")
    temporary.write_text(json.dumps(complete, indent=2), encoding="utf-8")
    temporary.replace(paths["complete"])
    if paths["checkpoint"].is_file():
        paths["checkpoint"].unlink()
    return complete


In [ ]:
def run_ga_scenario(scenario):
    if scenario not in FINAL_SCENARIOS:
        raise ValueError(f"Not a prespecified final GA scenario: {scenario}")
    action = prepare_scenario(scenario)
    if action == "skip":
        print("Valid completed result retained without overwrite:", SCENARIO_LABELS[scenario])
        return json.loads(scenario_paths(scenario)["complete"].read_text(encoding="utf-8"))

    restore_trainable(MODEL, FROZEN_BASELINE_STATE)
    assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256
    try:
        history, summary = train_ga(MODEL, scenario, resume=(action == "resume"))
        evidence = evaluate_selected_ga(MODEL, scenario, summary)
        result = publish_ga_result(MODEL, scenario, history, summary, evidence)
        print("COMPLETE:", SCENARIO_LABELS[scenario])
        return result
    finally:
        restore_trainable(MODEL, FROZEN_BASELINE_STATE)
        MODEL.zero_grad(set_to_none=True)
        clear_device_cache()
        assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256


### 8.7 Final Preflight Before Paid Updates


In [ ]:
preflight_rows = []
for scenario in FINAL_SCENARIOS:
    full_dir = FULL_RESULTS / scenario
    checks = {
        "Forget membership exact": (
            set(scenario_sets[scenario]["training_forget"]["assessment_id"])
            == authoritative_forget_ids(scenario)
        ),
        "Full Retraining COMPLETE exists": (full_dir / "COMPLETE.json").is_file(),
        "Full Retraining forget probabilities exist": (
            full_dir / "forget_set_probabilities.csv"
        ).is_file(),
    }
    for check_name, passed in checks.items():
        preflight_rows.append({
            "Scenario": SCENARIO_LABELS[scenario],
            "Check": check_name,
            "Passed": passed,
        })

preflight = pd.DataFrame(preflight_rows)
display(preflight)
if not preflight["Passed"].all():
    raise RuntimeError("GA preflight failed. Paid unlearning has not started.")
assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256
print("GA preflight: PASS")


## 9. Final Gradient Ascent Runs

Run these cells one at a time and verify `COMPLETE.json` before starting the next. They are intentionally unexecuted in this build.


### 9.1 Recipient Withdrawal

Expected training forget rows: **426**.


In [ ]:
recipient_result = run_ga_scenario(
    "recipient_withdrawal"
)


### 9.2 Invalid Consent

Expected training forget rows: **4,148**.


In [ ]:
invalid_result = run_ga_scenario(
    "invalid_consent"
)


### 9.3 Retention Expiry

Expected training forget rows: **6,262**.


In [ ]:
retention_result = run_ga_scenario(
    "retention_expiry"
)


## 10. Gradient Ascent Results

This table reads valid completed saved results only. Display rounding never alters raw files.


In [ ]:
saved_rows = []
for scenario in FINAL_SCENARIOS:
    paths = scenario_paths(scenario)
    if not paths["complete"].is_file():
        continue
    complete = json.loads(paths["complete"].read_text(encoding="utf-8"))
    if not valid_complete_payload(complete, scenario):
        continue
    metrics = pd.read_csv(paths["result_dir"] / "retained_test_metrics.csv").iloc[0]
    saved_rows.append({
        "Scenario": SCENARIO_LABELS[scenario],
        "Forget Rows": complete["training_forget_rows"],
        "Selected Epoch": complete["selected_epoch"],
        "Eligible Updated Model?": complete["eligible_updated_model"],
        "PR-AUC": metrics["pr_auc"],
        "Balanced Accuracy": metrics["balanced_accuracy"],
        "BCE": metrics["binary_cross_entropy"],
        "F1": metrics["f1"],
        "AUROC": metrics["auroc"],
        "KS Statistic": complete["ks_statistic"],
        "KS p-value": complete["ks_p_value"],
        "GA Runtime": complete["ga_total_required_method_seconds"],
        "Full Retraining Runtime": complete["full_retraining_training_seconds"],
        "Speed-up": complete["formal_speed_up"],
    })

final_results = pd.DataFrame(saved_rows)
display(final_results.round(4))
print(f"Completed saved scenarios: {len(final_results)}/{len(FINAL_SCENARIOS)}")


## 11. Findings and Limitations

> **Findings will be calculated from completed saved scenario results.**

Interpretation must keep scenario-level outcomes, retained utility, forgetting evidence, efficiency, and deletion-request size distinct.


In [ ]:
for scenario in FINAL_SCENARIOS:
    label = SCENARIO_LABELS[scenario]
    row = final_results.loc[final_results["Scenario"].eq(label)] if not final_results.empty else pd.DataFrame()
    heading = f"### {label} Result"
    if row.empty:
        message = "Pending a valid completed saved result."
    else:
        result = row.iloc[0]
        message = (
            f"Selected epoch: {int(result['Selected Epoch'])}; "
            f"eligible updated model: {bool(result['Eligible Updated Model?'])}; "
            f"retained-test PR-AUC: {result['PR-AUC']:.4f}; "
            f"KS statistic: {result['KS Statistic']:.4f}; "
            f"required GA runtime: {result['GA Runtime']:.1f} seconds."
        )
    display(Markdown(f"{heading}\n\n{message}"))

display(Markdown(
    "### Gradient Ascent Utility\nUse saved retained-test metrics only.\n\n"
    "### Gradient Ascent Forgetting\nUse saved Truth Ratio and KS evidence only.\n\n"
    "### Gradient Ascent Efficiency\nUse the documented timing boundaries.\n\n"
    "### Effect of Deletion-Request Size\nCompare only after all three results exist."
))


### 11.1 Limitations

- GA updates task-specific LoRA/head parameters rather than Qwen's original pretrained base.
- Gradient Ascent can be unstable because it deliberately reverses the forget objective.
- The 5% retained-validation rule constrains utility damage but does not guarantee perfect retention.
- The safety boundary is a predefined experimental rule, not proof of optimality.
- Only one Qwen model family is evaluated.
- The dataset is synthetic.
- Tabular-to-text serialisation adds computational overhead.
- Runtime is hardware dependent.
- Full Retraining is a behavioural reference.
- Truth Ratio and KS are behavioural evidence rather than proof of exact erasure or legal compliance.
